# Notebook 12: EF-Primary Motion Head Ablation

This notebook trains two EF-primary variants initialized from the existing segmentation-primary multitask checkpoint:

1. EF-primary ConvLSTM with segmentation as a low-weight auxiliary task.
2. EF-primary ConvLSTM with the same EF/segmentation setup plus an explicit optical-flow motion auxiliary head.

The motion head attaches to per-timestep fused bidirectional ConvLSTM bottleneck features. Existing model modules and parameter names are preserved so the notebook-11 multitask checkpoint remains compatible; only `motion_head.*` is newly initialized for the motion-supervised model.


## Motion-Head Attachment Plan

The base bidirectional ConvLSTM U-Net encodes every sampled frame into bottleneck features, runs a forward ConvLSTM and a backward ConvLSTM, fuses the two target-aligned hidden states, and decodes only the center target mask. Here we add a notebook-local method that stores the forward and backward hidden state for every sampled time index, fuses each pair with the existing `bidirectional_fusion`, and returns a tensor shaped `[B, T, C, H_b, W_b]`.

The EF and segmentation heads continue to consume only the target feature `H_target`. The motion head consumes consecutive fused features using `concat(H_t, H_{t+1}, H_{t+1} - H_t)` and predicts low-resolution optical flow `[u, v]` for each transition. Because the inherited module names are unchanged, loading the existing multitask checkpoint should match all pretrained keys; the only expected missing keys are the new `motion_head.*` weights.


## Setup


In [ ]:
from __future__ import annotations

from pathlib import Path
from contextlib import nullcontext
import json
import math
import os
import random
import sys
from typing import Any

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

def autocast_context(enabled: bool):
    if enabled and torch.cuda.is_available():
        if hasattr(torch, "amp") and hasattr(torch.amp, "autocast"):
            return torch.amp.autocast("cuda", enabled=True)
        return torch.cuda.amp.autocast(enabled=True)
    return nullcontext()


def make_grad_scaler(enabled: bool):
    if hasattr(torch, "amp") and hasattr(torch.amp, "GradScaler"):
        return torch.amp.GradScaler("cuda", enabled=enabled)
    return torch.cuda.amp.GradScaler(enabled=enabled)


def first_existing_path(candidates):
    cleaned = [candidate for candidate in candidates if candidate]
    for candidate in cleaned:
        path = Path(candidate)
        if path.exists():
            return path
    return Path(cleaned[-1])


PROJECT_ROOT = first_existing_path([
    os.environ.get("PROJECT_ROOT"),
    "/kaggle/input/echonet-temporal-xai",
    "/kaggle/input/src-updated",
    "/kaggle/working/Echonet_temporal_XAI",
    Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd(),
])
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.bidirectional_convlstm_unet import BidirectionalConvLSTMUNet
from src.dataset import EchoNetTemporalDataset, load_temporal_metadata, split_by_echonet_filelist
from src.temporal_train import get_temporal_loss, segmentation_metrics
from src.utils import load_echonet_tables, set_seed

RAW_DIR = Path(os.environ.get("ECHONET_RAW_DIR", PROJECT_ROOT / "data" / "raw" / "EchoNet-Dynamic"))
PROCESSED_DIR = Path(os.environ.get("ECHONET_PROCESSED_DIR", PROJECT_ROOT / "data" / "processed"))
VIDEOS_DIR = RAW_DIR / "Videos"
MULTITASK_CHECKPOINT_DIR = Path(os.environ.get(
    "MULTITASK_CHECKPOINT_DIR",
    PROJECT_ROOT / "outputs" / "runs" / "multitask_segmentation_ef_07_20" / "checkpoints",
))
RUN_DIR = Path(os.environ.get(
    "RUN_DIR",
    "/kaggle/working/outputs/runs/ef_primary_motion_head" if Path("/kaggle/working").exists() else PROJECT_ROOT / "outputs" / "runs" / "ef_primary_motion_head",
))
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
MANIFEST_DIR = RUN_DIR / "manifests"
FIGURES_DIR = RUN_DIR / "figures"
# Optical-flow targets are large. On Kaggle, keep this cache under /kaggle/temp so it
# does not count toward the saved /kaggle/working output quota.
FLOW_CACHE_DIR = Path(os.environ.get(
    "FLOW_CACHE_DIR",
    "/kaggle/temp/ef_primary_motion_flow" if Path("/kaggle").exists() else RUN_DIR / "flow_cache",
))
QUAL_DIR = FIGURES_DIR / "qualitative_examples"

for directory in [RUN_DIR, CHECKPOINT_DIR, MANIFEST_DIR, FIGURES_DIR, QUAL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)
FLOW_CACHE_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Raw EchoNet directory: {RAW_DIR}")
print(f"Processed directory: {PROCESSED_DIR}")
print(f"Multitask checkpoint directory: {MULTITASK_CHECKPOINT_DIR}")
print(f"Output directory: {RUN_DIR}")
print(f"Flow cache directory: {FLOW_CACHE_DIR}")



## Configuration


In [ ]:
RUN_MODE = "smoke"  # change to "full" for the complete Kaggle run
NUM_FRAMES_BEFORE = 11
NUM_FRAMES_AFTER = 11
TEMPORAL_STRIDE = 2
TARGET_IDX = NUM_FRAMES_BEFORE
SEQUENCE_LENGTH = NUM_FRAMES_BEFORE + 1 + NUM_FRAMES_AFTER
IMAGE_SIZE = (112, 112)
CHANNELS = (16, 32, 64, 128)
THRESHOLD = 0.5

SMOKE_CONFIG = {
    "run_mode": "smoke",
    "seed": 42,
    "batch_size": 2,
    "num_workers": 2,
    "max_train_samples": 16,
    "max_val_samples": 8,
    "max_test_samples": 8,
    "stage1_epochs": 1,
    "stage2_epochs": 1,
    "stage3_epochs": 0,
    "lambda_seg": 0.1,
    "lambda_motion": 0.1,
    "ef_head_lr": 1e-4,
    "motion_head_lr": 1e-4,
    "temporal_lr": 3e-5,
    "upper_encoder_lr": 1e-5,
    "weight_decay": 1e-5,
    "ef_hidden_dim": 128,
    "dropout": 0.1,
    "motion_hidden_channels": 64,
    "mixed_precision": True,
    "resume": True,
    "save_last_checkpoint": False,
    "qualitative_example_count": 4,
}

FULL_CONFIG = {
    "run_mode": "full",
    "seed": 42,
    "batch_size": 4,
    "num_workers": 2,
    "max_train_samples": None,
    "max_val_samples": None,
    "max_test_samples": None,
    "stage1_epochs": 5,
    "stage2_epochs": 10,
    "stage3_epochs": 3,
    "lambda_seg": 0.1,
    "lambda_motion": 0.1,
    "ef_head_lr": 1e-4,
    "motion_head_lr": 1e-4,
    "temporal_lr": 3e-5,
    "upper_encoder_lr": 1e-5,
    "weight_decay": 1e-5,
    "ef_hidden_dim": 128,
    "dropout": 0.1,
    "motion_hidden_channels": 64,
    "mixed_precision": True,
    "resume": True,
    "save_last_checkpoint": False,
    "qualitative_example_count": 12,
}

config = SMOKE_CONFIG if RUN_MODE == "smoke" else FULL_CONFIG
config.update({
    "num_frames_before": NUM_FRAMES_BEFORE,
    "num_frames_after": NUM_FRAMES_AFTER,
    "temporal_stride": TEMPORAL_STRIDE,
    "target_idx": TARGET_IDX,
    "sequence_length": SEQUENCE_LENGTH,
    "image_size": list(IMAGE_SIZE),
    "channels": list(CHANNELS),
    "threshold": THRESHOLD,
})
set_seed(config["seed"])
with (RUN_DIR / "config.json").open("w", encoding="utf-8") as file:
    json.dump(config, file, indent=2)
config


## Load EchoNet Labels and Official Splits


In [ ]:
metadata_path = PROCESSED_DIR / "metadata.csv"
assert metadata_path.exists(), f"Missing processed metadata: {metadata_path}"
assert (RAW_DIR / "FileList.csv").exists(), f"Missing FileList.csv under {RAW_DIR}"
assert VIDEOS_DIR.exists(), f"Missing Videos directory: {VIDEOS_DIR}"

samples = load_temporal_metadata(metadata_path)
file_list, volume_tracings = load_echonet_tables(RAW_DIR)
assert "EF" in file_list.columns, "FileList.csv must contain EF labels."

echo_table = file_list.copy()
echo_table["video_stem"] = echo_table["FileName"].astype(str).map(lambda x: Path(x).stem)
ef_lookup = dict(zip(echo_table["video_stem"], echo_table["EF"].astype(float)))

samples_with_ef = []
for sample in samples:
    item = dict(sample)
    video_stem = Path(str(item["video_id"])).stem
    if video_stem in ef_lookup and pd.notna(ef_lookup[video_stem]):
        item["ef"] = float(ef_lookup[video_stem])
        samples_with_ef.append(item)
assert samples_with_ef, "No processed samples could be matched to EF labels."

train_samples, val_samples, test_samples = split_by_echonet_filelist(samples_with_ef, file_list)
full_split_counts = {"train": len(train_samples), "validation": len(val_samples), "test": len(test_samples)}
assert min(full_split_counts.values()) > 0, f"Empty split after EF merge: {full_split_counts}"

if config["max_train_samples"] is not None:
    train_samples = train_samples[:config["max_train_samples"]]
if config["max_val_samples"] is not None:
    val_samples = val_samples[:config["max_val_samples"]]
if config["max_test_samples"] is not None:
    test_samples = test_samples[:config["max_test_samples"]]

print(f"Processed samples with EF: {len(samples_with_ef):,}")
print(f"Official full split counts: {full_split_counts}")
print(f"Active train/val/test: {len(train_samples):,} / {len(val_samples):,} / {len(test_samples):,}")



## ED/ES Sequence Coverage


In [ ]:
def find_explicit_ed_es_columns(file_list_df: pd.DataFrame) -> tuple[str | None, str | None]:
    ed_candidates = ["EDFrame", "FrameED", "ED_Frame", "ED_frame", "EDFrameIndex", "ED_frame_idx", "ed_frame", "ed_frame_idx"]
    es_candidates = ["ESFrame", "FrameES", "ES_Frame", "ES_frame", "ESFrameIndex", "ES_frame_idx", "es_frame", "es_frame_idx"]
    lower_to_actual = {str(col).lower(): col for col in file_list_df.columns}
    ed_col = next((lower_to_actual[name.lower()] for name in ed_candidates if name.lower() in lower_to_actual), None)
    es_col = next((lower_to_actual[name.lower()] for name in es_candidates if name.lower() in lower_to_actual), None)
    return ed_col, es_col


def mask_area_lookup(samples_with_ef: list[dict[str, Any]]) -> dict[tuple[str, int], float]:
    areas = {}
    for sample in tqdm(samples_with_ef, desc="measure processed mask areas", leave=False):
        mask = cv2.imread(str(sample["mask"]), cv2.IMREAD_GRAYSCALE)
        if mask is None:
            continue
        key = (Path(str(sample["video_id"])).stem, int(sample["frame_idx"]))
        areas[key] = float((mask > 0).sum())
    return areas


def raw_tracing_frame_lookup(tracings_df: pd.DataFrame) -> dict[str, list[int]]:
    required = {"FileName", "Frame"}
    missing = required.difference(tracings_df.columns)
    if missing:
        raise ValueError(f"VolumeTracings.csv is missing columns: {sorted(missing)}")
    tracing_table = tracings_df.copy()
    tracing_table["video_stem"] = tracing_table["FileName"].astype(str).map(lambda x: Path(x).stem)
    return tracing_table.groupby("video_stem")["Frame"].apply(lambda x: sorted(set(int(v) for v in x))).to_dict()


def build_ed_es_lookup(
    samples_with_ef: list[dict[str, Any]],
    file_list_df: pd.DataFrame,
    tracings_df: pd.DataFrame,
) -> tuple[dict[str, dict[str, Any]], str]:
    ed_col, es_col = find_explicit_ed_es_columns(file_list_df)
    if ed_col is not None and es_col is not None and "FileName" in file_list_df.columns:
        lookup = {}
        for _, row in file_list_df.iterrows():
            stem = Path(str(row["FileName"])).stem
            if pd.notna(row[ed_col]) and pd.notna(row[es_col]):
                lookup[stem] = {
                    "ed_frame_idx": int(row[ed_col]),
                    "es_frame_idx": int(row[es_col]),
                    "source": f"explicit_filelist_columns:{ed_col},{es_col}",
                }
        return lookup, "explicit_filelist_ed_es_frame_columns"

    tracing_frames = raw_tracing_frame_lookup(tracings_df)
    areas = mask_area_lookup(samples_with_ef)
    lookup = {}
    for video_id, frames in tracing_frames.items():
        if len(frames) < 2:
            continue
        frame_areas = [(frame, areas.get((video_id, int(frame)), np.nan)) for frame in frames]
        finite = [(frame, area) for frame, area in frame_areas if np.isfinite(area)]
        if len(finite) >= 2:
            # ED has the larger LV mask area; ES has the smaller LV mask area.
            ed_frame = int(max(finite, key=lambda item: item[1])[0])
            es_frame = int(min(finite, key=lambda item: item[1])[0])
            source = "raw_volume_tracings_frames_with_processed_mask_area_phase_assignment"
        else:
            # VolumeTracings still provides the annotated frame indices, but not phase names.
            # This fallback keeps coverage available while making the phase assignment explicit.
            ed_frame = int(frames[0])
            es_frame = int(frames[-1])
            source = "raw_volume_tracings_frames_fallback_sorted_phase_assignment"
        lookup[video_id] = {"ed_frame_idx": ed_frame, "es_frame_idx": es_frame, "source": source}
    return lookup, "raw_volume_tracings_frame_indices"


def sampled_indices(target_frame_idx: int) -> list[int]:
    return [target_frame_idx + offset * TEMPORAL_STRIDE for offset in range(-NUM_FRAMES_BEFORE, NUM_FRAMES_AFTER + 1)]

ed_es_lookup, ed_es_source = build_ed_es_lookup(samples_with_ef, file_list, volume_tracings)
phase_lookup = {}
coverage_rows = []
for sample in samples_with_ef:
    sample_id = str(sample["id"])
    video_id = Path(str(sample["video_id"])).stem
    target_frame = int(sample["frame_idx"])
    info = ed_es_lookup.get(video_id, {})
    ed_frame = info.get("ed_frame_idx")
    es_frame = info.get("es_frame_idx")
    sampled = sampled_indices(target_frame)
    contains_ed = ed_frame is not None and int(ed_frame) in sampled
    contains_es = es_frame is not None and int(es_frame) in sampled
    contains_both = contains_ed and contains_es
    if ed_frame is not None and target_frame == int(ed_frame):
        phase = "ED"
    elif es_frame is not None and target_frame == int(es_frame):
        phase = "ES"
    else:
        phase = "other_annotated"
    phase_lookup[sample_id] = phase
    coverage_rows.append({
        "sample_id": sample_id,
        "video_id": video_id,
        "target_frame_idx": target_frame,
        "phase": phase,
        "phase_source": info.get("source", "unavailable"),
        "ed_frame_idx": ed_frame,
        "es_frame_idx": es_frame,
        "ed_es_distance_frames": abs(int(es_frame) - int(ed_frame)) if ed_frame is not None and es_frame is not None else np.nan,
        "contains_ed_exact_sampled": bool(contains_ed),
        "contains_es_exact_sampled": bool(contains_es),
        "contains_both_ed_es_exact_sampled": bool(contains_both),
    })
coverage_df = pd.DataFrame(coverage_rows)
coverage_summary = {
    "sample_count": int(len(coverage_df)),
    "ed_es_source": ed_es_source,
    "phase_assignment_sources": coverage_df["phase_source"].value_counts(dropna=False).to_dict(),
    "percentage_contains_ed_exact_sampled": float(100.0 * coverage_df["contains_ed_exact_sampled"].mean()),
    "percentage_contains_es_exact_sampled": float(100.0 * coverage_df["contains_es_exact_sampled"].mean()),
    "percentage_contains_both_ed_es_exact_sampled": float(100.0 * coverage_df["contains_both_ed_es_exact_sampled"].mean()),
    "ed_es_distance_min_frames": float(coverage_df["ed_es_distance_frames"].min()),
    "ed_es_distance_median_frames": float(coverage_df["ed_es_distance_frames"].median()),
    "ed_es_distance_mean_frames": float(coverage_df["ed_es_distance_frames"].mean()),
    "ed_es_distance_max_frames": float(coverage_df["ed_es_distance_frames"].max()),
}
coverage_df.to_csv(MANIFEST_DIR / "ed_es_sequence_coverage.csv", index=False)
with (MANIFEST_DIR / "ed_es_sequence_coverage_summary.json").open("w", encoding="utf-8") as file:
    json.dump(coverage_summary, file, indent=2)
print(json.dumps(coverage_summary, indent=2))


## Dataset Wrappers


In [ ]:
class EchoNetTemporalEFMotionDataset(torch.utils.data.Dataset):
    def __init__(self, base_dataset: EchoNetTemporalDataset, ef_mean: float, ef_std: float, flow_cache_dir: Path | None = None) -> None:
        self.base_dataset = base_dataset
        self.ef_mean = float(ef_mean)
        self.ef_std = float(ef_std)
        self.flow_cache_dir = flow_cache_dir

    def __len__(self) -> int:
        return len(self.base_dataset)

    def _flow_path(self, sample_id: str) -> Path:
        safe_id = sample_id.replace("/", "_")
        return self.flow_cache_dir / f"{safe_id}.npz"

    def __getitem__(self, idx: int) -> dict[str, Any]:
        item = dict(self.base_dataset[idx])
        sample = self.base_dataset.samples[idx]
        ef = float(sample["ef"])
        item["ef"] = torch.tensor(ef, dtype=torch.float32)
        item["ef_normalized"] = torch.tensor((ef - self.ef_mean) / self.ef_std, dtype=torch.float32)
        item["phase"] = phase_lookup.get(str(item["id"]), "unknown")
        if self.flow_cache_dir is not None:
            flow_path = self._flow_path(str(item["id"]))
            with np.load(flow_path, allow_pickle=False) as data:
                flow = data["flow_uv"].astype(np.float32)
            item["flow_uv_fullres"] = torch.from_numpy(flow)
        return item


def make_base_dataset(samples: list[dict[str, Any]], augment: bool) -> EchoNetTemporalDataset:
    return EchoNetTemporalDataset(
        samples,
        videos_dir=VIDEOS_DIR,
        num_frames_before=config["num_frames_before"],
        num_frames_after=config["num_frames_after"],
        temporal_stride=config["temporal_stride"],
        image_size=tuple(config["image_size"]),
        augment=augment,
    )

ef_values_train = np.array([float(sample["ef"]) for sample in train_samples], dtype=np.float32)
ef_mean = float(ef_values_train.mean())
ef_std = float(ef_values_train.std(ddof=0))
assert ef_std > 0, "Training EF standard deviation is zero; cannot normalize EF."
print(f"EF normalization: mean={ef_mean:.3f}, std={ef_std:.3f}")


## Optical Flow Cache


In [ ]:
def compute_farneback_flow_pair(frame0: np.ndarray, frame1: np.ndarray) -> np.ndarray:
    prev = np.clip(frame0 * 255.0, 0, 255).astype(np.uint8)
    nxt = np.clip(frame1 * 255.0, 0, 255).astype(np.uint8)
    flow = cv2.calcOpticalFlowFarneback(
        prev, nxt, None,
        pyr_scale=0.5,
        levels=3,
        winsize=15,
        iterations=3,
        poly_n=5,
        poly_sigma=1.2,
        flags=0,
    )
    return flow.transpose(2, 0, 1).astype(np.float32)


def cache_flow_for_samples(samples_for_cache: list[dict[str, Any]], split: str) -> pd.DataFrame:
    dataset = make_base_dataset(samples_for_cache, augment=False)
    rows = []
    for idx in tqdm(range(len(dataset)), desc=f"cache flow {split}"):
        item = dataset[idx]
        sample_id = str(item["id"])
        safe_id = sample_id.replace("/", "_")
        out_path = FLOW_CACHE_DIR / f"{safe_id}.npz"
        if out_path.exists():
            rows.append({"split": split, "sample_id": sample_id, "video_id": str(item["video_id"]), "flow_path": str(out_path), "cached": True})
            continue
        sequence = item["sequence"].squeeze(1).numpy().astype(np.float32)
        flows = [compute_farneback_flow_pair(sequence[t], sequence[t + 1]) for t in range(SEQUENCE_LENGTH - 1)]
        flow_uv = np.stack(flows, axis=0).astype(np.float32)
        np.savez_compressed(
            out_path,
            flow_uv=flow_uv,
            frame_indices=item["frame_indices"].numpy().astype(np.int32),
            video_id=np.array(str(item["video_id"])),
            sample_id=np.array(sample_id),
        )
        rows.append({"split": split, "sample_id": sample_id, "video_id": str(item["video_id"]), "flow_path": str(out_path), "cached": False})
    return pd.DataFrame(rows)

flow_manifest_parts = [
    cache_flow_for_samples(train_samples, "train"),
    cache_flow_for_samples(val_samples, "validation"),
    cache_flow_for_samples(test_samples, "test"),
]
flow_manifest_df = pd.concat(flow_manifest_parts, ignore_index=True)
flow_manifest_df.to_csv(MANIFEST_DIR / "flow_cache_manifest.csv", index=False)
print(flow_manifest_df["cached"].value_counts(dropna=False))


## DataLoaders


In [ ]:
train_base = make_base_dataset(train_samples, augment=False)
val_base = make_base_dataset(val_samples, augment=False)
test_base = make_base_dataset(test_samples, augment=False)
# Keep all sequences deterministic: cached optical flow is computed from these exact sampled frames.
train_dataset = EchoNetTemporalEFMotionDataset(train_base, ef_mean, ef_std, FLOW_CACHE_DIR)
val_dataset = EchoNetTemporalEFMotionDataset(val_base, ef_mean, ef_std, FLOW_CACHE_DIR)
test_dataset = EchoNetTemporalEFMotionDataset(test_base, ef_mean, ef_std, FLOW_CACHE_DIR)

loader_kwargs = {
    "batch_size": config["batch_size"],
    "num_workers": config["num_workers"],
    "pin_memory": torch.cuda.is_available(),
    "persistent_workers": config["num_workers"] > 0,
}
train_loader = DataLoader(train_dataset, shuffle=True, **loader_kwargs)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_kwargs)

sample = train_dataset[0]
assert sample["sequence"].shape == (SEQUENCE_LENGTH, 1, *IMAGE_SIZE)
assert sample["mask"].shape == (1, *IMAGE_SIZE)
assert sample["flow_uv_fullres"].shape == (SEQUENCE_LENGTH - 1, 2, *IMAGE_SIZE)
print(f"Sequence: {tuple(sample['sequence'].shape)} | flow: {tuple(sample['flow_uv_fullres'].shape)} | mask: {tuple(sample['mask'].shape)}")



## EF-Primary Models and Motion Head


In [ ]:
class EFPrimaryConvLSTM(BidirectionalConvLSTMUNet):
    def __init__(self, *args, ef_hidden_dim: int = 128, dropout: float = 0.1, **kwargs) -> None:
        super().__init__(*args, **kwargs)
        bottleneck_channels = self.bidirectional_fusion[0].out_channels
        self.ef_pool = nn.AdaptiveAvgPool2d(1)
        self.ef_head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(bottleneck_channels, ef_hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(ef_hidden_dim, 1),
        )

    @staticmethod
    def _run_temporal_sequence(cell, features: list[torch.Tensor], order: list[int]) -> dict[int, torch.Tensor]:
        state = None
        hidden_by_index = {}
        for time_idx in order:
            encoded = features[time_idx]
            if state is None:
                state = cell.init_state(encoded)
            state = cell(encoded, state)
            hidden_by_index[time_idx] = state[0]
        return hidden_by_index

    def temporal_features(self, sequence: torch.Tensor) -> tuple[torch.Tensor, tuple[torch.Tensor, torch.Tensor, torch.Tensor]]:
        self._validate_sequence(sequence)
        bottlenecks, target_skips = self._encode_sequence(sequence)
        fwd = self._run_temporal_sequence(self.forward_temporal_bottleneck, bottlenecks, list(range(self.expected_sequence_length)))
        bwd = self._run_temporal_sequence(self.backward_temporal_bottleneck, bottlenecks, list(range(self.expected_sequence_length - 1, -1, -1)))
        fused = [self.bidirectional_fusion(torch.cat([fwd[t], bwd[t]], dim=1)) for t in range(self.expected_sequence_length)]
        return torch.stack(fused, dim=1), target_skips

    def decode_segmentation(self, fused_target: torch.Tensor, target_skips: tuple[torch.Tensor, torch.Tensor, torch.Tensor]) -> torch.Tensor:
        skip1, skip2, skip3 = target_skips
        x = self.decoder3(fused_target, skip3)
        x = self.decoder2(x, skip2)
        x = self.decoder1(x, skip1)
        return self.output(x)

    def forward(self, sequence: torch.Tensor, return_temporal: bool = False) -> dict[str, torch.Tensor]:
        fused_all, target_skips = self.temporal_features(sequence)
        fused_target = fused_all[:, self.target_idx]
        seg_logits = self.decode_segmentation(fused_target, target_skips)
        ef_normalized = self.ef_head(self.ef_pool(fused_target)).squeeze(1)
        out = {"seg_logits": seg_logits, "ef_normalized": ef_normalized}
        if return_temporal:
            out["temporal_features"] = fused_all
        return out


class MotionHead(nn.Module):
    def __init__(self, channels: int, hidden_channels: int = 64) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3 * channels, hidden_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(hidden_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden_channels, hidden_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(hidden_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden_channels, 2, kernel_size=1),
        )

    def forward(self, temporal_features: torch.Tensor) -> torch.Tensor:
        pairs = []
        for t in range(temporal_features.shape[1] - 1):
            h0 = temporal_features[:, t]
            h1 = temporal_features[:, t + 1]
            pairs.append(self.net(torch.cat([h0, h1, h1 - h0], dim=1)))
        return torch.stack(pairs, dim=1)


class EFPrimaryMotionConvLSTM(EFPrimaryConvLSTM):
    def __init__(self, *args, motion_hidden_channels: int = 64, **kwargs) -> None:
        super().__init__(*args, **kwargs)
        bottleneck_channels = self.bidirectional_fusion[0].out_channels
        self.motion_head = MotionHead(bottleneck_channels, hidden_channels=motion_hidden_channels)

    def forward(self, sequence: torch.Tensor, return_temporal: bool = False) -> dict[str, torch.Tensor]:
        out = super().forward(sequence, return_temporal=True)
        out["flow_pred"] = self.motion_head(out["temporal_features"])
        if not return_temporal:
            out.pop("temporal_features")
        return out


def build_model(with_motion: bool) -> nn.Module:
    cls = EFPrimaryMotionConvLSTM if with_motion else EFPrimaryConvLSTM
    kwargs = {
        "in_channels": 1,
        "out_channels": 1,
        "channels": tuple(config["channels"]),
        "num_frames_before": config["num_frames_before"],
        "num_frames_after": config["num_frames_after"],
        "ef_hidden_dim": config["ef_hidden_dim"],
        "dropout": config["dropout"],
    }
    if with_motion:
        kwargs["motion_hidden_channels"] = config["motion_hidden_channels"]
    return cls(**kwargs).to(device)

model_ef_primary = build_model(with_motion=False)
model_motion = build_model(with_motion=True)


## Checkpoint Loading


In [ ]:
def select_checkpoint(checkpoint_dir: Path) -> Path:
    for name in ["best_model.pt", "best_val_dice_model.pt", "final_model.pt"]:
        candidate = checkpoint_dir / name
        if candidate.exists():
            return candidate
    candidates = sorted(path for path in checkpoint_dir.iterdir() if path.suffix in {".pt", ".pth", ".ckpt"})
    if not candidates:
        raise FileNotFoundError(f"No checkpoint found in {checkpoint_dir}")
    return candidates[0]


def load_compatible_checkpoint(model_to_load: nn.Module, checkpoint_path: Path, expected_missing_prefixes: tuple[str, ...]) -> dict[str, Any]:
    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    state_dict = checkpoint.get("model_state_dict", checkpoint.get("state_dict", checkpoint))
    if any(key.startswith("module.") for key in state_dict):
        state_dict = {key.removeprefix("module."): value for key, value in state_dict.items()}
    model_state = model_to_load.state_dict()
    compatible = {}
    mismatched = []
    for key, value in state_dict.items():
        if key in model_state and tuple(value.shape) == tuple(model_state[key].shape):
            compatible[key] = value
        elif key in model_state:
            mismatched.append({"key": key, "checkpoint_shape": tuple(value.shape), "model_shape": tuple(model_state[key].shape)})
    result = model_to_load.load_state_dict(compatible, strict=False)
    missing = list(result.missing_keys)
    unexpected = [key for key in state_dict if key not in model_state]
    unexpected_non_ignored = [key for key in unexpected if not key.startswith("motion_head")]
    bad_missing = [key for key in missing if not key.startswith(expected_missing_prefixes)]
    assert not mismatched, f"Mismatched checkpoint tensors: {mismatched[:5]}"
    assert not bad_missing, f"Unexpected missing keys: {bad_missing}"
    assert not unexpected_non_ignored, f"Unexpected checkpoint keys: {unexpected_non_ignored[:10]}"
    return {"checkpoint_path": str(checkpoint_path), "missing_keys": missing, "unexpected_keys": unexpected, "mismatched_keys": mismatched}

MULTITASK_CHECKPOINT_PATH = select_checkpoint(MULTITASK_CHECKPOINT_DIR)
load_report = {
    "ef_primary": load_compatible_checkpoint(model_ef_primary, MULTITASK_CHECKPOINT_PATH, expected_missing_prefixes=()),
    "ef_primary_motion": load_compatible_checkpoint(model_motion, MULTITASK_CHECKPOINT_PATH, expected_missing_prefixes=("motion_head",)),
}
with (MANIFEST_DIR / "checkpoint_load_report.json").open("w", encoding="utf-8") as file:
    json.dump(load_report, file, indent=2)
print(json.dumps(load_report, indent=2))


## Shape and Loss Validation


In [ ]:
seg_loss_fn = get_temporal_loss()
ef_loss_fn = nn.SmoothL1Loss()
motion_loss_fn = nn.SmoothL1Loss()


def denormalize_ef(x: torch.Tensor | np.ndarray) -> torch.Tensor | np.ndarray:
    return x * ef_std + ef_mean


def resize_flow_to_prediction(flow_fullres: torch.Tensor, pred_hw: tuple[int, int]) -> torch.Tensor:
    b, transitions, channels, in_h, in_w = flow_fullres.shape
    out_h, out_w = pred_hw
    flat = flow_fullres.reshape(b * transitions, channels, in_h, in_w)
    resized = F.interpolate(flat, size=pred_hw, mode="bilinear", align_corners=False)
    resized[:, 0] *= out_w / max(in_w, 1)
    resized[:, 1] *= out_h / max(in_h, 1)
    return resized.reshape(b, transitions, channels, out_h, out_w)

sample_batch = next(iter(train_loader))
sequence = sample_batch["sequence"].to(device)
mask = sample_batch["mask"].to(device)
ef_target = sample_batch["ef_normalized"].to(device)
flow_target_full = sample_batch["flow_uv_fullres"].to(device)
with torch.no_grad():
    out_ef = model_ef_primary(sequence, return_temporal=True)
    out_motion = model_motion(sequence, return_temporal=True)
flow_target = resize_flow_to_prediction(flow_target_full, out_motion["flow_pred"].shape[-2:])
assert out_ef["seg_logits"].shape == mask.shape
assert out_ef["ef_normalized"].shape == ef_target.shape
assert out_ef["temporal_features"].shape[1] == SEQUENCE_LENGTH
assert out_motion["flow_pred"].shape == flow_target.shape
loss_check = ef_loss_fn(out_motion["ef_normalized"], ef_target) + config["lambda_seg"] * seg_loss_fn(out_motion["seg_logits"], mask) + config["lambda_motion"] * motion_loss_fn(out_motion["flow_pred"], flow_target)
assert torch.isfinite(loss_check)
print({
    "seg_logits": tuple(out_motion["seg_logits"].shape),
    "ef_pred": tuple(out_motion["ef_normalized"].shape),
    "temporal_features": tuple(out_motion["temporal_features"].shape),
    "flow_pred": tuple(out_motion["flow_pred"].shape),
    "flow_target": tuple(flow_target.shape),
    "loss_check": float(loss_check.detach().cpu()),
})


## Training Utilities


In [ ]:
def endpoint_error(pred: np.ndarray, target: np.ndarray) -> float:
    pred = np.asarray(pred, dtype=np.float32)
    target = np.asarray(target, dtype=np.float32)
    return float(np.sqrt(((pred - target) ** 2).sum(axis=2)).mean())


def ef_metrics(pred_percent: np.ndarray, target_percent: np.ndarray) -> dict[str, float]:
    pred_percent = np.asarray(pred_percent, dtype=np.float64)
    target_percent = np.asarray(target_percent, dtype=np.float64)
    errors = pred_percent - target_percent
    mae = float(np.mean(np.abs(errors))) if len(errors) else float("nan")
    rmse = float(np.sqrt(np.mean(errors ** 2))) if len(errors) else float("nan")
    corr = float(np.corrcoef(pred_percent, target_percent)[0, 1]) if len(errors) > 1 and np.std(pred_percent) > 1e-8 and np.std(target_percent) > 1e-8 else float("nan")
    return {"ef_mae": mae, "ef_rmse": rmse, "ef_pearson": corr}


def set_trainable(model_to_train: nn.Module, stage: int, with_motion: bool) -> None:
    for param in model_to_train.parameters():
        param.requires_grad = False
    for param in model_to_train.ef_head.parameters():
        param.requires_grad = True
    if with_motion:
        for param in model_to_train.motion_head.parameters():
            param.requires_grad = True
    if stage >= 2:
        for module in [model_to_train.forward_temporal_bottleneck, model_to_train.backward_temporal_bottleneck, model_to_train.bidirectional_fusion]:
            for param in module.parameters():
                param.requires_grad = True
    if stage >= 3:
        for module in [model_to_train.encoder3, model_to_train.bottleneck_encoder]:
            for param in module.parameters():
                param.requires_grad = True


def build_optimizer(model_to_train: nn.Module, with_motion: bool, stage: int) -> torch.optim.Optimizer:
    groups = []
    groups.append({"params": [p for p in model_to_train.ef_head.parameters() if p.requires_grad], "lr": config["ef_head_lr"], "name": "ef_head"})
    if with_motion:
        groups.append({"params": [p for p in model_to_train.motion_head.parameters() if p.requires_grad], "lr": config["motion_head_lr"], "name": "motion_head"})
    if stage >= 2:
        temporal_params = []
        for module in [model_to_train.forward_temporal_bottleneck, model_to_train.backward_temporal_bottleneck, model_to_train.bidirectional_fusion]:
            temporal_params.extend([p for p in module.parameters() if p.requires_grad])
        groups.append({"params": temporal_params, "lr": config["temporal_lr"], "name": "temporal"})
    if stage >= 3:
        upper_params = []
        for module in [model_to_train.encoder3, model_to_train.bottleneck_encoder]:
            upper_params.extend([p for p in module.parameters() if p.requires_grad])
        groups.append({"params": upper_params, "lr": config["upper_encoder_lr"], "name": "upper_encoder"})
    groups = [group for group in groups if group["params"]]
    return torch.optim.AdamW(groups, weight_decay=config["weight_decay"])


def compute_losses(out: dict[str, torch.Tensor], batch: dict[str, Any], with_motion: bool) -> dict[str, torch.Tensor]:
    masks = batch["mask"].to(device, non_blocking=True)
    ef_target = batch["ef_normalized"].to(device, non_blocking=True)
    ef_loss = ef_loss_fn(out["ef_normalized"], ef_target)
    seg_loss = seg_loss_fn(out["seg_logits"], masks)
    total = ef_loss + config["lambda_seg"] * seg_loss
    losses = {"total_loss": total, "ef_loss": ef_loss, "seg_loss": seg_loss, "weighted_seg_loss": config["lambda_seg"] * seg_loss}
    if with_motion:
        flow_target = resize_flow_to_prediction(batch["flow_uv_fullres"].to(device, non_blocking=True), out["flow_pred"].shape[-2:])
        motion_loss = motion_loss_fn(out["flow_pred"], flow_target)
        losses["motion_loss"] = motion_loss
        losses["weighted_motion_loss"] = config["lambda_motion"] * motion_loss
        losses["total_loss"] = losses["total_loss"] + losses["weighted_motion_loss"]
    else:
        losses["motion_loss"] = torch.tensor(float("nan"), device=device)
        losses["weighted_motion_loss"] = torch.tensor(float("nan"), device=device)
    return losses


def mean_ignore_all_nan(values: list[float]) -> float:
    array = np.asarray(values, dtype=np.float64)
    if array.size == 0 or not np.isfinite(array).any():
        return float("nan")
    return float(np.nanmean(array))


def aggregate_rows(rows: list[dict[str, Any]], prefix: str) -> dict[str, float]:
    total_n = sum(row["n"] for row in rows)
    dice = np.concatenate([row["dice"] for row in rows]) if rows else np.array([])
    iou = np.concatenate([row["iou"] for row in rows]) if rows else np.array([])
    ef_pred = np.concatenate([row["ef_pred"] for row in rows]) if rows else np.array([])
    ef_true = np.concatenate([row["ef_true"] for row in rows]) if rows else np.array([])
    metrics = {
        f"{prefix}_total_loss": sum(row["total_loss"] * row["n"] for row in rows) / max(total_n, 1),
        f"{prefix}_ef_loss": sum(row["ef_loss"] * row["n"] for row in rows) / max(total_n, 1),
        f"{prefix}_seg_loss": sum(row["seg_loss"] * row["n"] for row in rows) / max(total_n, 1),
        f"{prefix}_motion_loss": sum(row["motion_loss"] * row["n"] for row in rows) / max(total_n, 1),
        f"{prefix}_dice": float(np.mean(dice)) if len(dice) else float("nan"),
        f"{prefix}_iou": float(np.mean(iou)) if len(iou) else float("nan"),
        f"{prefix}_motion_epe": mean_ignore_all_nan([row["motion_epe"] for row in rows]),
    }
    metrics.update({f"{prefix}_{key}": value for key, value in ef_metrics(ef_pred, ef_true).items()})
    return metrics



## Train and Validate One Model


In [ ]:
def run_epoch(model_to_train: nn.Module, loader: DataLoader, optimizer=None, scaler=None, with_motion: bool = False, train: bool = False) -> dict[str, float]:
    model_to_train.train(train)
    rows = []
    for batch in tqdm(loader, desc="train" if train else "eval", leave=False):
        sequences = batch["sequence"].to(device, non_blocking=True)
        if train:
            optimizer.zero_grad(set_to_none=True)
        autocast_enabled = bool(config["mixed_precision"] and torch.cuda.is_available())
        with autocast_context(autocast_enabled):
            out = model_to_train(sequences)
            losses = compute_losses(out, batch, with_motion=with_motion)
        if train:
            if scaler is not None and autocast_enabled:
                scaler.scale(losses["total_loss"]).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                losses["total_loss"].backward()
                optimizer.step()
        with torch.no_grad():
            masks = batch["mask"].to(device, non_blocking=True)
            dice, iou = segmentation_metrics(out["seg_logits"], masks, threshold=THRESHOLD)
            ef_pred = denormalize_ef(out["ef_normalized"].detach().cpu()).numpy()
            ef_true = batch["ef"].detach().cpu().numpy()
            if with_motion:
                flow_target = resize_flow_to_prediction(batch["flow_uv_fullres"].to(device), out["flow_pred"].shape[-2:])
                motion_epe = endpoint_error(out["flow_pred"].detach().cpu().numpy(), flow_target.detach().cpu().numpy())
            else:
                motion_epe = float("nan")
        rows.append({
            "n": int(sequences.shape[0]),
            "total_loss": float(losses["total_loss"].detach().cpu()),
            "ef_loss": float(losses["ef_loss"].detach().cpu()),
            "seg_loss": float(losses["seg_loss"].detach().cpu()),
            "motion_loss": float(losses["motion_loss"].detach().cpu()) if with_motion else float("nan"),
            "dice": dice.detach().cpu().numpy(),
            "iou": iou.detach().cpu().numpy(),
            "ef_pred": ef_pred,
            "ef_true": ef_true,
            "motion_epe": motion_epe,
        })
    return aggregate_rows(rows, "train" if train else "val")


def training_schedule() -> list[int]:
    stages = []
    for stage, epochs in [(1, config["stage1_epochs"]), (2, config["stage2_epochs"]), (3, config["stage3_epochs"])]:
        stages.extend([stage] * int(epochs))
    return stages


def train_model(model_name: str, model_to_train: nn.Module, with_motion: bool) -> pd.DataFrame:
    model_dir = CHECKPOINT_DIR / model_name
    model_dir.mkdir(parents=True, exist_ok=True)
    model_config = {**config, "model_name": model_name, "with_motion_head": bool(with_motion)}
    with (model_dir / "config.json").open("w", encoding="utf-8") as file:
        json.dump(model_config, file, indent=2)

    history_path = MANIFEST_DIR / f"{model_name}_history.csv"
    history = []
    completed_epochs = 0
    best_val_ef_mae = float("inf")
    last_path = model_dir / "last.pt"
    if config.get("resume", True) and last_path.exists() and history_path.exists():
        checkpoint = torch.load(last_path, map_location=device)
        model_to_train.load_state_dict(checkpoint["model_state_dict"])
        history = pd.read_csv(history_path).to_dict("records")
        completed_epochs = int(len(history))
        if history:
            best_val_ef_mae = float(pd.DataFrame(history)["val_ef_mae"].min())
        print(f"Resuming {model_name} after {completed_epochs} completed epochs.")

    schedule = training_schedule()
    current_optimizer = None
    current_scaler = None
    current_stage = None
    for global_epoch, stage in enumerate(schedule, start=1):
        if global_epoch <= completed_epochs:
            continue
        if stage != current_stage:
            current_stage = stage
            set_trainable(model_to_train, stage=stage, with_motion=with_motion)
            current_optimizer = build_optimizer(model_to_train, with_motion=with_motion, stage=stage)
            current_scaler = make_grad_scaler(enabled=bool(config["mixed_precision"] and torch.cuda.is_available()))
        train_metrics = run_epoch(model_to_train, train_loader, optimizer=current_optimizer, scaler=current_scaler, with_motion=with_motion, train=True)
        with torch.no_grad():
            val_metrics = run_epoch(model_to_train, val_loader, with_motion=with_motion, train=False)
        row = {"model_name": model_name, "stage": stage, "epoch": global_epoch, **train_metrics, **val_metrics}
        history.append(row)
        pd.DataFrame(history).to_csv(history_path, index=False)
        checkpoint_payload = {
            "model_state_dict": model_to_train.state_dict(),
            "optimizer_state_dict": current_optimizer.state_dict(),
            "epoch": global_epoch,
            "stage": stage,
            "metrics": row,
            "config": model_config,
            "ef_mean": ef_mean,
            "ef_std": ef_std,
        }
        best_payload = {
            "model_state_dict": model_to_train.state_dict(),
            "epoch": global_epoch,
            "stage": stage,
            "metrics": row,
            "config": model_config,
            "ef_mean": ef_mean,
            "ef_std": ef_std,
        }
        if config.get("save_last_checkpoint", False):
            torch.save(checkpoint_payload, last_path)
        if row["val_ef_mae"] < best_val_ef_mae:
            best_val_ef_mae = row["val_ef_mae"]
            torch.save(best_payload, model_dir / "best_ef_mae.pt")
        print(f"{model_name} stage={stage} epoch={global_epoch} val_EF_MAE={row['val_ef_mae']:.2f} val_dice={row['val_dice']:.4f} val_motion_epe={row['val_motion_epe']:.4f}")
    return pd.DataFrame(history)

history_ef_primary = train_model("ef_primary", model_ef_primary, with_motion=False)
history_motion = train_model("ef_primary_motion", model_motion, with_motion=True)



## Temporal Perturbation Conditions


In [ ]:
PERTURBATION_SEED = 42


def normal_sequence(sequences: torch.Tensor) -> torch.Tensor:
    return sequences


def zero_motion_sequence(sequences: torch.Tensor) -> torch.Tensor:
    return sequences[:, TARGET_IDX:TARGET_IDX + 1].repeat(1, SEQUENCE_LENGTH, 1, 1, 1).contiguous()


def context_shuffled_target_fixed_sequence(sequences: torch.Tensor) -> torch.Tensor:
    generator = torch.Generator(device="cpu")
    generator.manual_seed(PERTURBATION_SEED)
    context_indices = [idx for idx in range(SEQUENCE_LENGTH) if idx != TARGET_IDX]
    shuffled_context = torch.tensor(context_indices, device=sequences.device)[torch.randperm(len(context_indices), generator=generator).to(sequences.device)]
    order = torch.empty(SEQUENCE_LENGTH, dtype=torch.long, device=sequences.device)
    order[TARGET_IDX] = TARGET_IDX
    order[torch.tensor(context_indices, device=sequences.device)] = shuffled_context
    assert int(order[TARGET_IDX].item()) == TARGET_IDX
    return sequences[:, order].contiguous()


def reversed_sequence(sequences: torch.Tensor) -> torch.Tensor:
    return torch.flip(sequences, dims=(1,)).contiguous()


def block_shuffled_sequence(sequences: torch.Tensor, block_size: int = 4) -> torch.Tensor:
    generator = torch.Generator(device="cpu")
    generator.manual_seed(PERTURBATION_SEED)
    blocks = [list(range(start, min(start + block_size, SEQUENCE_LENGTH))) for start in range(0, SEQUENCE_LENGTH, block_size)]
    order_blocks = torch.randperm(len(blocks), generator=generator).tolist()
    order = [idx for block_id in order_blocks for idx in blocks[block_id]]
    return sequences[:, torch.tensor(order, device=sequences.device)].contiguous()


def blurred_target_sequence(sequences: torch.Tensor) -> torch.Tensor:
    out = sequences.clone()
    b, _, c, h, w = out.shape
    target = out[:, TARGET_IDX].reshape(b * c, 1, h, w)
    kernel = torch.ones((1, 1, 7, 7), device=out.device, dtype=out.dtype) / 49.0
    blurred = F.conv2d(target, kernel, padding=3).reshape(b, c, h, w)
    out[:, TARGET_IDX] = blurred
    return out

PERTURBATION_CONDITIONS = [
    {"name": "normal_chronological", "transform": normal_sequence},
    {"name": "zero_motion_target_repeated", "transform": zero_motion_sequence},
    {"name": "context_shuffled_target_fixed", "transform": context_shuffled_target_fixed_sequence},
    {"name": "sequence_reversed", "transform": reversed_sequence},
    {"name": "block_shuffled_sequence", "transform": block_shuffled_sequence},
    {"name": "blurred_target_frame", "transform": blurred_target_sequence},
]


## Evaluation


In [ ]:
def load_best_weights(model_to_eval: nn.Module, model_name: str) -> nn.Module:
    checkpoint_path = CHECKPOINT_DIR / model_name / "best_ef_mae.pt"
    if checkpoint_path.exists():
        checkpoint = torch.load(checkpoint_path, map_location=device)
        model_to_eval.load_state_dict(checkpoint["model_state_dict"])
    model_to_eval.eval()
    return model_to_eval

original_multitask = build_model(with_motion=False)
load_compatible_checkpoint(original_multitask, MULTITASK_CHECKPOINT_PATH, expected_missing_prefixes=())
model_ef_primary = load_best_weights(model_ef_primary, "ef_primary")
model_motion = load_best_weights(model_motion, "ef_primary_motion")

@torch.no_grad()
def evaluate_condition(model_to_eval: nn.Module, model_name: str, with_motion: bool, condition: dict[str, Any]) -> tuple[pd.DataFrame, dict[str, float]]:
    rows = []
    agg_rows = []
    for batch in tqdm(test_loader, desc=f"{model_name} {condition['name']}", leave=False):
        sequences = condition["transform"](batch["sequence"].to(device, non_blocking=True))
        masks = batch["mask"].to(device, non_blocking=True)
        out = model_to_eval(sequences)
        losses = compute_losses(out, batch, with_motion=with_motion)
        dice, iou = segmentation_metrics(out["seg_logits"], masks, threshold=THRESHOLD)
        ef_pred = denormalize_ef(out["ef_normalized"].detach().cpu()).numpy()
        ef_true = batch["ef"].detach().cpu().numpy()
        if with_motion:
            flow_target = resize_flow_to_prediction(batch["flow_uv_fullres"].to(device), out["flow_pred"].shape[-2:])
            motion_epe = endpoint_error(out["flow_pred"].detach().cpu().numpy(), flow_target.detach().cpu().numpy())
        else:
            motion_epe = float("nan")
        agg_rows.append({
            "n": int(sequences.shape[0]),
            "total_loss": float(losses["total_loss"].detach().cpu()),
            "ef_loss": float(losses["ef_loss"].detach().cpu()),
            "seg_loss": float(losses["seg_loss"].detach().cpu()),
            "motion_loss": float(losses["motion_loss"].detach().cpu()) if with_motion else float("nan"),
            "dice": dice.detach().cpu().numpy(),
            "iou": iou.detach().cpu().numpy(),
            "ef_pred": ef_pred,
            "ef_true": ef_true,
            "motion_epe": motion_epe,
        })
        ids = [str(x) for x in batch["id"]]
        for i, sample_id in enumerate(ids):
            rows.append({
                "model_name": model_name,
                "condition": condition["name"],
                "sample_id": sample_id,
                "video_id": str(batch["video_id"][i]),
                "target_frame_idx": int(batch["frame_idx"][i]),
                "ef_true": float(ef_true[i]),
                "ef_pred": float(ef_pred[i]),
                "ef_error": float(ef_pred[i] - ef_true[i]),
                "dice": float(dice[i].detach().cpu()),
                "iou": float(iou[i].detach().cpu()),
            })
    metrics = aggregate_rows(agg_rows, model_name)
    summary = {
        "model_name": model_name,
        "condition": condition["name"],
        "ef_mae": metrics[f"{model_name}_ef_mae"],
        "ef_rmse": metrics[f"{model_name}_ef_rmse"],
        "ef_pearson": metrics[f"{model_name}_ef_pearson"],
        "dice": metrics[f"{model_name}_dice"],
        "iou": metrics[f"{model_name}_iou"],
        "motion_epe": metrics[f"{model_name}_motion_epe"],
    }
    return pd.DataFrame(rows), summary

model_specs = [
    {"name": "original_segmentation_primary_multitask", "model": original_multitask, "with_motion": False},
    {"name": "ef_primary", "model": model_ef_primary, "with_motion": False},
    {"name": "ef_primary_motion", "model": model_motion, "with_motion": True},
]
all_predictions = []
summary_rows = []
for spec in model_specs:
    for condition in PERTURBATION_CONDITIONS:
        pred_df, summary = evaluate_condition(spec["model"], spec["name"], spec["with_motion"], condition)
        all_predictions.append(pred_df)
        summary_rows.append(summary)

predictions_df = pd.concat(all_predictions, ignore_index=True)
summary_df = pd.DataFrame(summary_rows)
normal_lookup = summary_df[summary_df["condition"] == "normal_chronological"].set_index("model_name")
for metric in ["ef_mae", "ef_rmse", "dice", "iou", "motion_epe"]:
    summary_df[f"normal_{metric}"] = summary_df["model_name"].map(normal_lookup[metric].to_dict())
    summary_df[f"delta_{metric}_vs_normal"] = summary_df[metric] - summary_df[f"normal_{metric}"]
summary_df["dice_degradation_vs_normal"] = summary_df["normal_dice"] - summary_df["dice"]
summary_df.to_csv(MANIFEST_DIR / "model_perturbation_comparison.csv", index=False)
predictions_df.to_csv(MANIFEST_DIR / "model_perturbation_predictions.csv", index=False)
display(summary_df)


## Plots and Qualitative Examples


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for model_name, group in summary_df.groupby("model_name"):
    axes[0].plot(group["condition"], group["ef_mae"], marker="o", label=model_name)
    axes[1].plot(group["condition"], group["dice"], marker="o", label=model_name)
axes[0].set_title("EF MAE by perturbation")
axes[1].set_title("Dice by perturbation")
for ax in axes:
    ax.tick_params(axis="x", rotation=45)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "perturbation_comparison.png", dpi=150, bbox_inches="tight")
plt.close(fig)

@torch.no_grad()
def save_qualitative_examples(max_examples: int = 6) -> pd.DataFrame:
    rows = []
    saved = 0
    model_motion.eval()
    for batch in test_loader:
        sequences = batch["sequence"].to(device)
        masks = batch["mask"].to(device)
        ids = [str(x) for x in batch["id"]]
        for i, sample_id in enumerate(ids):
            panels = [("target", sequences[i, TARGET_IDX, 0].detach().cpu().numpy()), ("ground truth", masks[i, 0].detach().cpu().numpy())]
            for condition in PERTURBATION_CONDITIONS:
                seq = condition["transform"](sequences[i:i+1])
                out = model_motion(seq)
                pred = (torch.sigmoid(out["seg_logits"])[0, 0] >= THRESHOLD).float().detach().cpu().numpy()
                ef_pred = float(denormalize_ef(out["ef_normalized"].detach().cpu()).numpy()[0])
                panels.append((f"{condition['name']}\nEF={ef_pred:.1f}%", pred))
            cols = 4
            rows_n = int(math.ceil(len(panels) / cols))
            fig, axes = plt.subplots(rows_n, cols, figsize=(cols * 3.0, rows_n * 3.0), squeeze=False)
            for panel_idx, ax in enumerate(axes.flat):
                ax.axis("off")
                if panel_idx < len(panels):
                    title, image = panels[panel_idx]
                    ax.imshow(image, cmap="gray", vmin=0, vmax=1)
                    ax.set_title(title, fontsize=8)
            fig.suptitle(f"{sample_id} | true EF={float(batch['ef'][i]):.1f}%", fontsize=11)
            fig.tight_layout(rect=(0, 0, 1, 0.96))
            out_path = QUAL_DIR / f"{sample_id}_ef_primary_motion_conditions.png"
            fig.savefig(out_path, dpi=150, bbox_inches="tight")
            plt.close(fig)
            rows.append({"sample_id": sample_id, "figure_path": str(out_path.relative_to(RUN_DIR))})
            saved += 1
            if saved >= max_examples:
                return pd.DataFrame(rows)
    return pd.DataFrame(rows)

qualitative_df = save_qualitative_examples(config["qualitative_example_count"])
qualitative_df.to_csv(MANIFEST_DIR / "qualitative_examples.csv", index=False)
display(qualitative_df)


## Required Outputs


In [ ]:
required_outputs = [
    RUN_DIR / "config.json",
    MANIFEST_DIR / "checkpoint_load_report.json",
    MANIFEST_DIR / "flow_cache_manifest.csv",
    MANIFEST_DIR / "ed_es_sequence_coverage.csv",
    MANIFEST_DIR / "ed_es_sequence_coverage_summary.json",
    MANIFEST_DIR / "ef_primary_history.csv",
    MANIFEST_DIR / "ef_primary_motion_history.csv",
    MANIFEST_DIR / "model_perturbation_comparison.csv",
    MANIFEST_DIR / "model_perturbation_predictions.csv",
    CHECKPOINT_DIR / "ef_primary" / "best_ef_mae.pt",
    CHECKPOINT_DIR / "ef_primary_motion" / "best_ef_mae.pt",
    FIGURES_DIR / "perturbation_comparison.png",
]
missing = [str(path) for path in required_outputs if not path.exists()]
assert not missing, f"Missing expected outputs: {missing}"
print(f"All outputs saved under: {RUN_DIR}")
